# Validation — `kappa-lora-metamathqa-rank32`

**What this measures:** Target = `num_trainable_params` from the repo's own MetaMathQA harness — the exact protocol the requester's notebook uses to test contributions (run.py, Llama-3.2-3B, r=32, 5000 steps, GSM8K): κ-selection via `condition_number_top_fraction=0.5` (a default-off knob, so it needs a new experiment config that turns it ON) must cut full LoRA's exact 48,627,712 trainable params to ≤ 24,313,856 — the honest "halves" bar, since κ keeps 98 of 196 equally-counted modules whose per-module sizes range 131k–360k params (realized share can be 32.1–67.9%) — while GSM8K "test accuracy" guards the "without losing fit" half at the same-protocol reference 0.4632 − 0.02.

**How I read the claim:** The claim is that κ-LoRA's spectral targeting (condition_number_top_fraction=0.5, this PR's only behavioral knob) halves LoRA's trainable parameters without losing fit. Following the requester's notebook pattern, we add experiments/lora/llama-3.2-3B-rank32-kappa0.5 beside the published baseline and run method_comparison/MetaMathQA/run.py (Llama-3.2-3B, r=32, 196 candidate modules, 5000 steps, GSM8K), 3 seeds, comparing temporary_results JSONs against the published lora--llama-3.2-3B-rank32.json row. Support = num_trainable_params ≤ 24,313,856 (half of the geometry-derived 48,627,712) AND median test accuracy ≥ row − 0.02 (in-context floor 0.4432), with total_time ≤ 0.838×row as a secondary cost check. The key caveat: 'top 50% of matrices' is a module-count rule, and under this model's heterogeneous module sizes it yields 32.1%–67.9% of LoRA's parameters depending on which module types rank highest by κ — so the notebook must print the selected-module type mix, and an MLP-heavy selection above the halving threshold is a geometry effect, not a code bug. The paper's 4.5% memory claim is unmeasurable under this harness (no accelerator_memory_max field) and the paper's benchmarks are not GSM8K, so 'matching accuracy' is being re-established, not imported.

- ⚠️ 'Top 50% of matrices' is a module-count fraction; under Llama-3.2-3B's mixed module sizes (131k–360k params each) the realized parameter fraction is 32.1%–67.9%, exactly 50% only for a type-balanced selection — the paper's uniform 'halves the trainable parameter count' implicitly assumes balanced/uniform modules
- ⚠️ The paper's −4.5% memory claim has no field in the MetaMathQA harness (the notebook comparison prints trainable/valid/test/wall-clock only); accelerator_memory_max exists only in image-gen rows (lora/flux2-klein-default: 10,708,058,112, so the analogous threshold would be ≤ 10,226,215,517) — verifying it needs a secondary flux2-klein arm with the same knob
- ⚠️ The paper's benchmarks are not MetaMathQA/GSM8K and its 16.2%/4.5% figures are cross-benchmark averages; 'matching accuracy' must be re-established on this repo's 5000-step GSM8K protocol, not imported

**Target metric:** `num_trainable_params`

**Repository:** [mayorquinmachines/peft](https://github.com/mayorquinmachines/peft) at commit [`86fa642551e0`](https://github.com/mayorquinmachines/peft/commit/86fa642551e0494a23914a7dde51c61f629f231d)

**Benchmark:** the repository's own `method_comparison/MetaMathQA/run.py` over `experiments/lora/llama-3.2-3B-rank32-kappa0.5` — not a synthesized stand-in, so the numbers are comparable to what this repository publishes.

**Nothing here has been executed** — there are no outputs and no result is being claimed. Review the measurement, edit the configuration or criteria if it is wrong, then mention `@remyx validate` to run it on Remyx compute — or run the cells top to bottom yourself on a machine with a GPU.

In [ ]:
# Parameters (Remyx passes the commit it measures as `ref`)
variant = "feature"
ref = ""
seed = 0

## 1. Environment

A CUDA GPU is required; the published protocol peaks above 22 GB.

In [ ]:
!nvidia-smi -L
import sys, torch
print(f"python {sys.version.split()[0]} · torch {torch.__version__} · cuda {torch.cuda.is_available()}")

## 2. The code under test

Clone the repository and check out exactly the commit that was validated, then install it in editable mode so the harness imports this checkout. When this notebook runs on Remyx compute the checkout already exists at that commit, and this cell only confirms it.

In [ ]:
import os, subprocess, sys
REPO_URL = "https://github.com/mayorquinmachines/peft"
COMMIT = ref or "86fa642551e0494a23914a7dde51c61f629f231d"

def _sh(*cmd):
    return subprocess.run(cmd, check=True, text=True, capture_output=True).stdout.strip()

def _at_commit():
    try:
        return os.path.isdir(".git") and _sh("git", "rev-parse", "HEAD").startswith(COMMIT)
    except Exception:
        return False

if not _at_commit():
    if not os.path.isdir("repo"):
        _sh("git", "clone", "--quiet", REPO_URL, "repo")
    os.chdir("repo")
    _sh("git", "fetch", "--quiet", "--depth=1", "origin", COMMIT)
    _sh("git", "checkout", "--quiet", COMMIT)
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", "."], check=True)
ROOT = os.getcwd()
print(ROOT)
print(_sh("git", "log", "-1", "--oneline"))

## 3. Credentials

If the benchmark downloads gated models or datasets it needs a Hugging Face token. In Colab, store it as a secret named `HF_TOKEN`; elsewhere set the environment variable.

In [ ]:
import os
if not os.environ.get("HF_TOKEN"):
    try:
        from google.colab import userdata
        os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    except Exception:
        pass
print("HF_TOKEN set" if os.environ.get("HF_TOKEN") else "HF_TOKEN not set — gated downloads will fail")

## 4. The experiment configuration

The harness runs a method by its configuration directory. This validation points it at `experiments/lora/llama-3.2-3B-rank32-kappa0.5` (relative to `method_comparison/MetaMathQA`).

`method_comparison/MetaMathQA/experiments/lora/llama-3.2-3B-rank32-kappa0.5/config.json`:

```json
{
  "method": "lora",
  "r": 32,
  "lora_alpha": 32,
  "lora_dropout": 0.0,
  "bias": "none",
  "task_type": "CAUSAL_LM",
  "target_modules": ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
  "kappa": 0.5
}
```

In [ ]:
print(open(os.path.join(ROOT, "method_comparison/MetaMathQA/experiments/lora/llama-3.2-3B-rank32-kappa0.5/config.json")).read())

## 5. Confirm the change under test is what is loaded

The commit printed here must match the one checked out above.

In [ ]:
import importlib
print(_sh("git", "rev-parse", "HEAD"))

## 6. Run the benchmark

`method_comparison/MetaMathQA/run.py` over `experiments/lora/llama-3.2-3B-rank32-kappa0.5` — a directory of experiments runs each in turn; a single experiment runs once.

In [ ]:
os.chdir(os.path.join(ROOT, "method_comparison/MetaMathQA"))
import glob, importlib, runpy, sys, time
RUN_STARTED = time.time()
configs = sorted(glob.glob("experiments/lora/llama-3.2-3B-rank32-kappa0.5/*/")) or ["experiments/lora/llama-3.2-3B-rank32-kappa0.5"]
for cfg in configs:
    print(f"[remyx] {cfg}")
    sys.argv = ["run.py", cfg.rstrip("/")]
    runpy.run_path("run.py", run_name="__main__")

## 7. Read what the benchmark wrote

Results land under `temporary_results/lora--llama-3.2-3B-rank32-kappa0.5--*.json` (relative to `method_comparison/MetaMathQA`) — or wherever this harness writes for a non-default checkout; only a document written by the run above counts. The metrics the criteria are judged against are fields of that document.

In [ ]:
import glob, json, os
PATTERNS = ["temporary_results/lora--llama-3.2-3B-rank32-kappa0.5--*.json"]
paths = sorted((p for pat in PATTERNS for p in glob.glob(pat)), key=os.path.getmtime)
paths = [p for p in paths if os.path.getmtime(p) >= RUN_STARTED - 1]
if not paths:
    # Some harnesses write elsewhere depending on the checkout (peft uses
    # temporary_results/ off the main branch): any document this run wrote.
    paths = sorted((p for p in glob.glob("**/*.json", recursive=True)
                    if os.path.getmtime(p) >= RUN_STARTED - 1 and "experiments/" not in p),
                   key=os.path.getmtime)
assert paths, "the benchmark wrote no result document"
doc = json.load(open(paths[-1]))
print("result document:", paths[-1])

def find(obj, key):
    """Last value under `key` anywhere in the document ('test accuracy' matches test_accuracy)."""
    hit = None
    if isinstance(obj, dict):
        for k, v in obj.items():
            if str(k).replace(" ", "_") == key and isinstance(v, (int, float)):
                hit = v
            found = find(v, key)
            hit = found if found is not None else hit
    elif isinstance(obj, list):
        for item in obj:
            found = find(item, key)
            hit = found if found is not None else hit
    return hit

METRICS = ["num_trainable_params", "test accuracy", "valid accuracy", "total_time"]
observed = {name: find(doc, name) for name in METRICS}
print(json.dumps(observed, indent=2))

## 8. Against the criteria

Thresholds come from `.remyx/validation.yaml`, so a failing measurement reports rather than crashes. `baseline` is the published row this repository already ships for the comparison method.

In [ ]:
CRITERIA = [
    {
        "metric": "num_trainable_params",
        "direction": "<=",
        "threshold": 24313856,
        "baseline": 48627712
    },
    {
        "metric": "test accuracy",
        "direction": ">=",
        "threshold": 0.4432,
        "baseline": 0.4632
    },
    {
        "metric": "valid accuracy",
        "direction": ">=",
        "threshold": 0.48,
        "baseline": 0.5
    },
    {
        "metric": "total_time",
        "direction": "<=",
        "threshold": 1566.0,
        "baseline": null
    }
]

print(f"{'metric':<28}{'observed':>16}{'baseline':>16}  criterion")
for c in CRITERIA:
    v = observed.get(c["metric"])
    t = c["threshold"]
    ok = None if v is None or t is None else (v <= t if c["direction"] == "<=" else v >= t)
    mark = "?" if ok is None else ("PASS" if ok else "FAIL")
    fmt = lambda x: (f"{x:.6g}" if isinstance(x, float) else str(x))
    print(f"{c['metric']:<28}{fmt(v):>16}{fmt(c['baseline']):>16}  {c['direction']} {fmt(t)}  {mark}")

## 9. Report

One line, machine-readable — what Remyx records as this run's measurement.

In [ ]:
print(json.dumps(observed))

## 10. What the outcome means

- **All rows pass** → the claim holds at this protocol: `num_trainable_params` <= 24313856 with `test accuracy` >= 0.4432, `valid accuracy` >= 0.48, `total_time` <= 1566.0 holding.
- **`num_trainable_params` fails** → the change does not deliver what the claim says at this protocol.
- **A guardrail fails** → the target may be met at the cost of something the claim promised to keep; look at the run log before drawing a conclusion.
- **No result document** → the benchmark did not finish; the run cell above says why.

## Appendix — the criteria file

`.remyx/validation.yaml` as committed:

```yaml
model:
  provider: zai
benchmarks:
  - name: kappa-lora-metamathqa-rank32
    suite:
      harness:
        runner: method_comparison/MetaMathQA/run.py
        experiments: experiments/lora/llama-3.2-3B-rank32-kappa0.5
        results_glob: method_comparison/MetaMathQA/temporary_results/lora--llama-3.2-3B-rank32-kappa0.5--*.json
        method: lora
        smoke:
          # keys mirror the harness's own training-params file (the 5k/lr naming of the ia3/frod training_params.json siblings);
          # 30 steps + eval every 15 proves the whole pipeline (gated download, injection, train, GSM8K eval, result JSON) in minutes
          params_path: method_comparison/MetaMathQA/default_training_params.json
          overrides:
            max_steps: 30
            eval_steps: 15
      scorer: num_trainable_params
      metrics:
        # full LoRA r=32 all-linear on Llama-3.2-3B = 28*(2*6144 + 2*4096 + 3*11264)*32 = 48,627,712 params; "halves" -> <= exactly half = 24,313,856.
        # kappa keeps ceil(196*0.5)=98 modules by COUNT irrespective of size (131,072-360,448 params each), so the realized share
        # ranges 32.1% (attention-light) to 67.9% (MLP-heavy); this threshold is therefore the honest test of the "halves" claim, not a formality.
        - name: num_trainable_params
          direction: min
          threshold: 24313856
          role: target
        # guardrail floor: 0.4632 same-protocol GSM8K test-accuracy reference - 0.02 parity band (notebook user_resource);
        # baseline row (published lora rank32) is itself comfortably above it
        - name: "test accuracy"
          direction: max
          threshold: 0.4432
          role: guardrail
        # 0.5000 same-protocol valid-accuracy reference - 0.02
        - name: "valid accuracy"
          direction: max
          threshold: 0.48
          role: guardrail
        # 26.1-min (1566 s) A100 same-protocol reference run; paper's -16.2% would imply ~1312 s; cost role - reported, never a gate
        - name: total_time
          direction: min
          threshold: 1566.0
          role: cost
    baseline:
      source: method_comparison/MetaMathQA/results/lora--llama-3.2-3B-rank32.json
      values:
        num_trainable_params: 48627712
        "test accuracy": 0.4632
        "valid accuracy": 0.5
    compute:
      tier: gpu
      # reference same-protocol run: 26.1 min (1566 s) on A100 for 5000 steps + GSM8K valid/test; 2.3x headroom covers the
      # full-LoRA-sized per-step cost, the 196 one-time fp32 SVDs (up to 8192x3072) at injection, and eval generation
      timeout_s: 3600
    policy:
      guardrail_veto: true
    held_constant:
      - "base model meta-llama/Llama-3.2-3B (28 layers, hidden 3072), gated access via HF_TOKEN"
      - "5000 train steps on MetaMathQA, GSM8K valid + test eval, single run via the repo's own run.py harness"
      - "LoRA r=32, lora_alpha=32, lora_dropout=0.0, same 7 target module types as the published lora--llama-3.2-3B-rank32 row"
      - "no training_params.json in the new experiment dir so the harness default_training_params.json protocol applies unchanged"
      - "kappa selection computed once at injection on fp32 base weights; rank r, alpha and every other LoraConfig knob untouched by the diff"
    avoid:
      - "unpinned base-model revisions"
      - "gating on time or memory: the paper's -16.2% time / -4.5% memory are cross-benchmark averages and the MetaMathQA result rows carry no memory field"
      - "comparing against supertuning/deft/unilora rows: only lora--llama-3.2-3B-rank32 is protocol-comparable"
      - "reading accuracy deltas smaller than 0.02 as signal: the notebook's parity band is +/-0.02 on this single-run protocol"
      - "editing run.py or default_training_params.json to accommodate the experiment"
    provenance:
      num_trainable_params: "user_guidance: halves trainable params vs standard LoRA; threshold = 28*(2*6144+2*4096+3*11264)*32/2 exactly from Llama-3.2-3B geometry"
      "test accuracy": "user_resource:https://colab.research.google.com/drive/1z73-jtAGrq4HkjvorjFcZ77uMwdmWs56?usp=sharing"
      "valid accuracy": "user_resource:https://colab.research.google.com/drive/1z73-jtAGrq4HkjvorjFcZ77uMwdmWs56?usp=sharing"
      total_time: "user_resource:https://colab.research.google.com/drive/1z73-jtAGrq4HkjvorjFcZ77uMwdmWs56?usp=sharing"
      suite: "repo_runner:method_comparison/MetaMathQA/run.py"
      baseline: "published_corpus:method_comparison/MetaMathQA/results/lora--llama-3.2-3B-rank32.json"
      held_constant: "protocol_doc:method_comparison/MetaMathQA/README.md"
      compute: "user_resource:https://colab.research.google.com/drive/1z73-jtAGrq4HkjvorjFcZ77uMwdmWs56?usp=sharing"
loop:
  max_iterations: 8
  fix_code: true
```